# Advanced HEN Methods

**Learning outcome:** Apply advanced hen methods through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `solver`  
**Expected runtime:** 10 to 90 minutes  
**Optional extras:** `synthesis`

## Study question and data

**Study question:** How do alternative synthesis and network-improvement methods change the candidate design?

## Step 1: Run a deeper synthesis and watch it

This cell loads the four-stream Yee and Grossmann case and runs `heat_exchanger_network(depth=3)`. Search depth 3 adds the MILP pinch design and thermal derivative methods to the default depth 1, makes more pinch design networks and evolves them more widely; `width` would widen the dt_cont multipliers, which default to 0.75, 1.0 and 1.5.

A deeper search takes longer, so the cell follows it while it runs. It passes a `HeatExchangerNetworkRecord` as `record=` and runs the design on a background thread. Every network the search solves and verifies enters the record straight away, including each improvement an evolution search finds on its way, one entry per topology. Meanwhile the loop asks `record.changed_since(seen)` for what changed after the version it last saw; the answer is `None` when nothing did. Each changed entry goes into `arrivals` with the seconds since the start. `deeper` holds the finished result, and `deeper_ranking` is its best successful outcome.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

from OpenPinch import PinchProblem
from OpenPinch.analysis.heat_exchanger_networks.results.record import (
    HeatExchangerNetworkRecord,
)

problem = PinchProblem(
    "Four-stream-Yee-and-Grossmann-1990-1.json", project_name="Four Stream"
)
record = HeatExchangerNetworkRecord()
arrivals = []
seen = 0
started = time.perf_counter()
with ThreadPoolExecutor(max_workers=1) as background:
    search = background.submit(
        problem.design.heat_exchanger_network, depth=3, record=record
    )
    while not search.done() or record.version > seen:
        change = record.changed_since(seen)
        if change is None:
            time.sleep(0.2)
            continue
        seen = change["version"]
        for entry in change["entries"]:
            arrivals.append(
                {
                    "seconds": round(time.perf_counter() - started, 1),
                    "version": entry["version"],
                    "method": entry["method"],
                    "units": entry["unit_count"],
                    "total_annual_cost": round(entry["total_annual_cost"]),
                }
            )
    deeper = search.result()
deeper_ranking = deeper.top(1)

## Step 2: Read the live record

This cell tabulates the arrivals in the order the search found them (by record version; one poll lists its changes in rank order), with the cheapest total annual cost seen so far, so you can see when the search found its best network and how long it then spent confirming it. When the search finds a known topology again at a lower cost, that entry is replaced, so an early arrival can be missing from the final record. The finished result is the record: `ranked_networks` holds exactly the record's outcomes, cheapest first, and the record is `"complete"` with its cheapest entry `selected`. A user interface would poll `changed_since` the same way and merge the changed entries by `key`.

In [ ]:
import pandas as pd

arrival_table = pd.DataFrame(arrivals).sort_values("version", ignore_index=True)
arrival_table["cheapest_so_far"] = arrival_table["total_annual_cost"].cummin()

final = record.snapshot()
assert record.status == "complete"
assert deeper.result.ranked_networks == record.outcomes()
assert final["selected"] == final["entries"][0]["key"]
record_table = pd.DataFrame(final["entries"])[
    ["rank", "method", "unit_count", "total_annual_cost"]
]

## Step 3: Compare advanced design methods

This cell runs the design methods one at a time. `pinch_design()` runs the pinch design step alone: the rules-based (Linnhoff) method, which proposes networks from the CP and number-of-streams rules at the pinch, at the default depth 1. From depth 2 the MILP method runs too. `thermal_derivative()` is seeded with the pinch design network, and `network_evolution()` with the thermal-derivative network. Compare the `*_ranking` outputs.

In [ ]:
pinch_design = problem.design.pinch_design()
pinch_ranking = pinch_design.top(1)
with_milp = problem.design.pinch_design(options={"HENS_SEARCH_DEPTH": 2})
with_milp_ranking = with_milp.top(1)
thermal = problem.design.thermal_derivative((pinch_design.selected_network,))
thermal_ranking = thermal.top(1)
evolution = problem.design.network_evolution((thermal.selected_network,))
evolution_ranking = evolution.top(1)

## Step 4: Optimise duties on a hand-built network

This cell defines a network structure by hand: four recovery matches in two stages as `(hot stream, cold stream, stage)`, a heater on Milk Concentrate and a cooler on each hot stream. `optimise_duties()` keeps that structure and allocates duty (and stream splits) three ways. `min_utility` minimises utility use with at least 10 K approach on every exchanger. `min_area` minimises total area with hot utility capped at 1.5 times that minimum. `min_cost` minimises total annual cost, requiring only a 1 K approach. Each result is a normal design view, so `top()`, `grid()` and the utility totals work as for the synthesis methods.

In [ ]:
structure = {
    "recovery": [
        ("Raw Milk", "Milk Concentrate", 1),
        ("HT Flash", "Milk Concentrate", 1),
        ("Raw Milk", "CIP Water", 2),
        ("HT Flash", "CIP Water", 2),
    ],
    "heaters": ["Milk Concentrate"],
    "coolers": ["Raw Milk", "HT Flash"],
}
min_utility = problem.design.optimise_duties(
    structure, objective="utility", min_approach_temperature=10.0
)
min_area = problem.design.optimise_duties(
    structure,
    objective="area",
    min_approach_temperature=10.0,
    max_hot_utility=1.5 * min_utility.total_hot_utility,
)
min_cost = problem.design.optimise_duties(structure, objective="cost")
duty_rankings = [view.top(1) for view in (min_utility, min_area, min_cost)]

## Step 5: Retrofit an existing network

This cell loads the five-stream Ciric and Floudas retrofit case and describes the network as built: four recovery exchangers in two stages, a heater on C2 and coolers on H1 and H2, each with its installed area (m2) and plant tag. `retrofit()` rates that network at its installed areas, then makes one structural change at a time, up to `max_modifications`: here new matches, ranked by the utility their structure could reach with unlimited area, each solved with the added area its existing exchangers need. Existing area is sunk; added area costs the `COSTING_HX_AREA_*` law, a new exchanger `COSTING_HX_UNIT_COST`, and here an area addition $8,650 and a stream split $10,000. `retrofit.frontier` holds the best design for each number of structural changes, from the existing network (0) up; `retrofit_frontier` lists each one's hot utility (kW), net annual benefit ($/y), payback (years) and modifications, area additions included.

In [ ]:
retrofit_problem = PinchProblem(
    "Five-stream-Ciric-and-Floudas-1989.json", project_name="Five Stream Retrofit"
)
existing = {
    "recovery": [
        {"hot": "H3", "cold": "C2", "stage": 1, "area": 609.7, "id": "E1"},
        {"hot": "H3", "cold": "C1", "stage": 2, "area": 579.2, "id": "E2"},
        {"hot": "H1", "cold": "C1", "stage": 1, "area": 1008.5, "id": "E3"},
        {"hot": "H2", "cold": "C2", "stage": 2, "area": 117.96, "id": "E4"},
    ],
    "heaters": [{"stream": "C2", "area": 246.75, "id": "E7"}],
    "coolers": [
        {"stream": "H1", "area": 787.5, "id": "E5"},
        {"stream": "H2", "area": 104.6, "id": "E6"},
    ],
}
retrofit = retrofit_problem.design.retrofit(
    existing,
    max_modifications=3,
    area_addition_cost=8_650.0,
    split_cost=10_000.0,
)
retrofit_frontier = [
    {
        "structural_changes": sum(
            m.kind != "area_addition" for m in design.modifications
        ),
        "hot_utility_kW": round(design.hot_utility),
        "net_annual_benefit": round(design.net_annual_benefit),
        "payback_years": design.payback_years,
        "steps": [(m.kind, m.exchanger_id) for m in design.modifications],
    }
    for design in retrofit.frontier
]

Two more kinds of change are opt-in. With `relocation_cost` ($ per move, here the report's $15,000) a step may move an existing recovery exchanger, with its installed area, to another match or position. `enhancement_factors` maps an existing exchanger to the factor its heat transfer coefficient can be raised by, with tube inserts for example (1.3 here, in the range Akpomiemie and Smith report for twisted tapes); an insert costs `enhancement_cost` plus `enhancement_area_cost` per m2 of the exchanger's area, and each structure is also tried with inserts in place of added area. `upgraded_frontier` lists the same columns; compare its net annual benefit and capital with `retrofit_frontier`.

In [ ]:
upgraded = retrofit_problem.design.retrofit(
    existing,
    max_modifications=3,
    area_addition_cost=8_650.0,
    split_cost=10_000.0,
    relocation_cost=15_000.0,
    enhancement_factors=dict.fromkeys(("E1", "E2", "E3", "E4"), 1.3),
    enhancement_cost=500.0,
    enhancement_area_cost=10.0,
)
upgraded_frontier = [
    {
        "structural_changes": sum(
            m.kind not in {"area_addition", "enhancement"} for m in design.modifications
        ),
        "hot_utility_kW": round(design.hot_utility),
        "net_annual_benefit": round(design.net_annual_benefit),
        "capital": round(design.capital_cost),
        "steps": [(m.kind, m.exchanger_id) for m in design.modifications],
    }
    for design in upgraded.frontier
]

## Review the result

Compare the leading result from every method on the same stream and economic basis before attributing design differences to the search method.

In [ ]:
from IPython.display import display

display(arrival_table)
display(record_table)
display(deeper_ranking)
display(pinch_ranking)
display(with_milp_ranking)
display(thermal_ranking)
display(evolution_ranking)
display(duty_rankings)
display(retrofit_frontier)
display(upgraded_frontier)

## Interpret the result

Compare every method on the same streams, utility assumptions, and economic basis; record solver status with each result.

## Adapt this template

Start with one method and small search bounds, then pass only reviewed networks into derivative or evolution workflows.